<a href="https://colab.research.google.com/github/Ivan8Garcia/Proyectos-n8n-RAG-AgentesIA/blob/main/Agentes_con_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df= pd.read_csv("/content/datos_entregas.csv")
df.head()

,ID_pedido,años_experiencia_colaborador,clasificacion_colaborador,latitud_tienda,longitud_tienda,latitud_entrega,longitud_entrega,fecha_pedido,hora_pedido,hora_retirada,clima,trafico,vehiculo,area,categoria_producto,tiempo_entrega
0,ialx566343618,8,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Soleado,Alto,Motocicleta,Urbano,Ropa,120
1,akqg208421122,10,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Lluvioso,Congestion,Scooter,Metropolitano,Electrónicos,165
2,njpu434582536,16,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Tormenta,Bajo,Motocicleta,Urbano,Deportes,130
3,rjto796129700,8,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Soleado,Medio,Motocicleta,Metropolitano,Cosméticos,105
4,zguw716275638,11,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Nublado,Alto,Scooter,Metropolitano,Juguetes,150


In [2]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')

In [3]:
!pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.2 MB/s eta 0:00:00


In [7]:
from langchain_groq import ChatGroq

llm=ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.1-8b-instant", temperature=0)

In [10]:
ai_msg= llm.invoke("""
                   tengo  un data frame llamado "df" con las columnas "años_experiencia_colaborador" y "tiempo_entrega".
                   escribe el codigo python con la biblioteca pandas para calcular la correlacion entre las dos columnas.
                   devuelve la respuesta en markdown para el trecho de codigo python unicamente.
                   """)
print(ai_msg)

content='### Calcular la correlación entre dos columnas de un DataFrame con Pandas\n\n```python\nimport pandas as pd\n\n# Calcula la correlación entre las columnas "años_experiencia_colaborador" y "tiempo_entrega"\ncorrelacion = df[\'años_experiencia_colaborador\'].corr(df[\'tiempo_entrega\'])\n\nprint(f"La correlación entre \'años_experiencia_colaborador\' y \'tiempo_entrega\' es: {correlacion}")\n```\n\nO también puedes utilizar el método `corr()` del DataFrame para calcular la correlación entre todas las columnas:\n\n```python\ncorrelacion = df.corr()\n\nprint(correlacion)\n```\n\nEsto te mostrará una matriz de correlaciones entre todas las columnas del DataFrame.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 167, 'prompt_tokens': 106, 'total_tokens': 273, 'completion_time': 0.201702522, 'completion_tokens_details': None, 'prompt_time': 0.006245234, 'prompt_tokens_details': None, 'queue_time': 0.048634857, 'total_time': 0.207947756}, 'model_name': 'll

In [11]:
print(ai_msg.content)

### Calcular la correlación entre dos columnas de un DataFrame con Pandas

```python
import pandas as pd

# Calcula la correlación entre las columnas "años_experiencia_colaborador" y "tiempo_entrega"
correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])

print(f"La correlación entre 'años_experiencia_colaborador' y 'tiempo_entrega' es: {correlacion}")
```

O también puedes utilizar el método `corr()` del DataFrame para calcular la correlación entre todas las columnas:

```python
correlacion = df.corr()

print(correlacion)
```

Esto te mostrará una matriz de correlaciones entre todas las columnas del DataFrame.


In [14]:
correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])

print(f"La correlación entre 'años_experiencia_colaborador' y 'tiempo_entrega' es: {correlacion}")


La correlación entre 'años_experiencia_colaborador' y 'tiempo_entrega' es: -0.2516410755060904


#**PythonAstREPLTool**  

-Lee- Evalua- Imprime- Itera

In [16]:
!pip install langchain-experimental -q

In [20]:
from langchain_experimental.tools import PythonAstREPLTool

herramienta_py=PythonAstREPLTool(locals={"df":df})
herramienta_py.invoke("df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])")

np.float64(-0.2516410755060904)

In [32]:
llm_con_herramienta=llm.bind_tools([herramienta_py],tool_choice=herramienta_py.name)
respuesta= llm_con_herramienta.invoke("""
                                      tengo  un data frame llamado "df" con las columnas "años_experiencia_colaborador" y "tiempo_entrega".
                                      escribe el codigo python con la biblioteca pandas para calcular la correlacion entre las dos columnas.
                                      devuelve la respuesta en markdown para el trecho de codigo python unicamente, separando siempre cada comando utilizando ";".
                                      """)


respuesta

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=python_repl_ast>import pandas as pd; df = pd.DataFrame({\'años_experiencia_colaborador\': [1, 2, 3, 4, 5], \'tiempo_entrega\': [10, 20, 30, 40, 50]}); corr = df[\'años_experiencia_colaborador\'].corr(df[\'tiempo_entrega\']); print(f"La correlación entre las dos columnas es: {corr}")</function>'}}

In [29]:
from langchain_core.output_parsers.openai_tools import JsonOutputKeyToolsParser

parser= JsonOutputKeyToolsParser(key_name=herramienta_py.name, first_tool_only=True)

cadena= llm_con_herramienta | parser

respuesta= cadena.invoke("""
              tengo  un data frame llamado "df" con las columnas "años_experiencia_colaborador" y "tiempo_entrega".
              escribe el codigo python con la biblioteca pandas para calcular la correlacion entre las dos columnas.
              devuelve la respuesta en markdown para el trecho de codigo python unicamente,separando siempre cada comando utilizando ";".
              """)
respuesta["query"]

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=python_repl_ast>import pandas as pd; df = pd.DataFrame({\'años_experiencia_colaborador\': [1, 2, 3, 4, 5], \'tiempo_entrega\': [10, 20, 30, 40, 50]}); corr = df[\'años_experiencia_colaborador\'].corr(df[\'tiempo_entrega\']); print(f"La correlación entre las dos columnas es: {corr}")</function>'}}

In [ ]:
import pandas as pd; corr_coef = df['años_experiencia_colaborador'].corr(df['tiempo_entrega']); print(corr_coef)

In [ ]:
df.columns.to_list()

In [34]:
from langchain_core.prompts import ChatPromptTemplate

system = f"""
          Tienes acceso a un dataframe pandas `df`.
          Aqui esta la salida del comando `df.columns.to_list()`:

          ```
          {df.columns.to_list()}
          ```

          Dada una pregunta de usuario, escribe el código Python para responderla,
          Cada comando python generado SIEMPRE debe ser separado por ';'.
          Los únicos recursos que debes utilizar son las bibliotecas incorporadas Python y pandas.
          Retorna ÚNICAMENTE el código Python.
          """

prompt = ChatPromptTemplate.from_messages([("system",system),("human","{question}")])

In [ ]:
cadena = prompt | llm_con_herramienta | parser | herramienta_py

respuesta = cadena.invoke({"question":"Cuál es la correlación entre años experiencia del colaborador y tiempo de entrega?"})

print(respuesta)

In [ ]:
respuesta = cadena.invoke({"question":"Calcula el promedio de tiempo de entrega para cada clima?"})

print(respuesta)